In [1]:
import os
os.listdir("random_collect")

['akvfeewngf.png',
 'anskobyvlt.png',
 'armjzggldd.png',
 'bmntrmtoay.png',
 'bzzuxkqbma.png',
 'cduqnqsrbd.png',
 'cercoqqtig.png',
 'cknncmalmm.png',
 'cnzoitxarb.png',
 'czeacdgagr.png',
 'data.json',
 'dczarzehkw.png',
 'dhpjcckawa.png',
 'djqrbyohfc.png',
 'dzsevgrunb.png',
 'gimyobvonm.png',
 'gkueshlukn.png',
 'gswonbxsfk.png',
 'guwhgejnoe.png',
 'hggqxndprj.png',
 'hrxweoyozv.png',
 'hyzlpcmacl.png',
 'hzdyqlqaas.png',
 'hzidubpvqh.png',
 'ifwosdxstk.png',
 'iqtsddqole.png',
 'izchlxxraa.png',
 'jieclaarhr.png',
 'jpgulhchdq.png',
 'jyynbxlwrc.png',
 'knxjvtymmg.png',
 'mqxpqorgts.png',
 'nfflovfbkp.png',
 'nmvqabjora.png',
 'nykryksmsp.png',
 'ocqyyopgma.png',
 'ozxjfdxkrs.png',
 'pfastiebnk.png',
 'pipgplxwgl.png',
 'qbpkdfmjou.png',
 'qnnzmwgayo.png',
 'rggaujzhbf.png',
 'rqlqtftqxu.png',
 'saxeqpdicf.png',
 'skyuhiizck.png',
 'tzvfrlpvzp.png',
 'tzvicffmgf.png',
 'udcxkqrmoi.png',
 'uucmvynsny.png',
 'vianbzawzv.png',
 'viyeaqxwyd.png',
 'vufgovofmx.png',
 'wrwqcjehvg.png'

In [2]:
import json
with open("random_collect/data.json", "r") as f:
    data = json.load(f)

In [6]:
for i in os.listdir("random_collect"):
    if not i in [i["filename"]+".png" for i in data]:
        os.remove(os.path.join("random_collect", i))

In [ ]:
import torch
import sys
sys.path.append("..")
from src.sd3_pipeline import VSFStableDiffusion3Pipeline
import json
import judge
import wandb
import numpy as np
import dotenv
dotenv.load_dotenv()
import argparse

parser = argparse.ArgumentParser(description="Run NAG sweep")
parser.add_argument("--eval_later", action="store_true", help="Run evaluation later")
args = parser.parse_args()

model_id = "stabilityai/stable-diffusion-3.5-large-turbo"
pipe = VSFStableDiffusion3Pipeline.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
)
pipe.to("cuda")

with open("../prompts/test_prompts.json.new", "r") as f:
    dev_prompts = json.load(f)

def run(scale, offset):
    wandb.init(project="vsf-sweep")
    score = np.array([0, 0], dtype=int)
    total = 0
    for seed in range(1):
        for i in dev_prompts:
            image = pipe(
                i["prompt"],
                negative_prompt=i["missing_element"],
                guidance_scale=0.,
                scale=scale,
                offset=offset,
                num_inference_steps=8,
                generator=torch.Generator("cuda").manual_seed(seed),
            ).images[0]
            if not args.eval_later:
                delta = judge.vqa(image, i["question_1"], i["question_2"])
                score += delta
                total += 1
                from PIL import ImageDraw, ImageFont
                draw = ImageDraw.Draw(image)
                text = f"{delta[0]}, {delta[1]}, -: {i['missing_element']}"
                draw.text((10, 10), text, fill="white")
                wandb.log({"pos_score":score[0]/total, "neg_score":score[1]/total, "img": wandb.Image(image, caption=f"+: {i['prompt']}\n -: {i['missing_element']}")})
            else:
                wandb.log({"img": wandb.Image(image, caption=f"+: {i['prompt']}\n -: {i['missing_element']}")})

run(4.5, 0.2)


['.env',
 '.ipynb_checkpoints',
 '__pycache__',
 'ablation',
 'capybara.png',
 'eval',
 'input_flip.ipynb',
 'judge.py',
 'main.ipynb',
 'nag++.csv',
 'nag_original.csv',
 'nasa.csv',
 'none.csv',
 'output.mp4',
 'results_cfg',
 'results_nasa',
 'results_nasa.zip',
 'run.sh',
 'runtime',
 'select.ipynb',
 'sweep_nag.py',
 'sweep_vsf.py',
 'test_nag.py',
 'test_nag_original.py',
 'test_normal_flux.py',
 'test_vsf.py',
 'test_vsf_flux.py',
 'video_reward',
 'vsf.csv',
 'wandb']